# Phase 4: Modelling (Part 1: Baseline Random Forest)
**Client:** Crédit Nationale Azur
**Objective:** Build a baseline Random Forest classifier to predict personal loan acceptance.

In this notebook, we will load the cleaned data from Phase 3 using the correct relative directory paths. We will rigorously prepare the data to prevent leakage by applying encoding, feature selection, and standardisation strictly after the train/test split. Finally, we will evaluate the baseline model using metrics appropriate for imbalanced classes.

In [8]:
# Required Base Imports for Phase 4
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2

## 1. Load Data and Map Target Variable
We begin by loading our cleaned dataset from the `data` directory using `joblib`. As required by the schema, the target variable `personal_loan` must be mapped to binary integers (1/0).

In [9]:
# Load the cleaned data using relative paths and joblib
df = joblib.load('../data/cleaned_data.pkl')

# Map the target variable exactly as specified
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})

# Separate features (X) and target (y)
X = df.drop('personal_loan', axis=1)
y = df['personal_loan']

## 2. Train/Test Split
To strictly prevent data leakage, all transformations must occur after splitting the data. We use an 80/20 split and apply stratification to maintain the class imbalance ratio in both sets. We also use `.copy()` to avoid the Pandas `SettingWithCopyWarning`.

In [10]:
# Perform the 80/20 train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create explicit copies to prevent SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

## 3. Encode Categorical Features
We will use Pandas `pd.get_dummies` to encode our categorical variables while retaining the column headers. We apply this to both train and test sets, then align them to ensure identical column structures.

In [11]:
# Identify ALL categorical columns containing text
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

# Loop through and apply pd.get_dummies() exactly as per class notes
for col in categorical_cols:
    # Transform X_train 
    dummies_train = pd.get_dummies(X_train[col], prefix=col, dtype=int)
    X_train.drop([col], axis=1, inplace=True)
    X_train = pd.concat([X_train, dummies_train], axis=1)
    
    # Transform X_test strictly after to prevent data leakage
    dummies_test = pd.get_dummies(X_test[col], prefix=col, dtype=int)
    X_test.drop([col], axis=1, inplace=True)
    X_test = pd.concat([X_test, dummies_test], axis=1)

# Align columns to guarantee train and test sets match perfectly
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("All categorical variables successfully one-hot encoded.")

All categorical variables successfully one-hot encoded.


## 4. Feature Selection
We must select the top 7 features using `SelectKBest` and the `chi2` scoring function. 
*Note: This step must be performed BEFORE standardisation, as the `chi2` test cannot accept negative values.*

In [12]:
# Initialise SelectKBest with chi2 and k=7
selector = SelectKBest(score_func=chi2, k=7)

# Fit strictly on the training data, then transform both
X_train_selected_array = selector.fit_transform(X_train, y_train)
X_test_selected_array = selector.transform(X_test)

# Cross-reference the headless Numpy array back to Pandas column names
selected_mask = selector.get_support()
selected_features = X_train.columns[selected_mask]

# Reconstruct DataFrames with the selected features
X_train_selected = pd.DataFrame(X_train_selected_array, columns=selected_features, index=X_train.index)
X_test_selected = pd.DataFrame(X_test_selected_array, columns=selected_features, index=X_test.index)

print(f"Selected Features: {list(selected_features)}")

Selected Features: ['yrs_experience', 'income', 'mortgage_amt', 'fixed_deposit_acct', 'education_level_Advanced or Professional', 'education_level_Graduate', 'education_level_Undergraduate']


## 5. Standardisation
We will scale the continuous features that survived the selection process using `StandardScaler`. We fit the scaler strictly on the training data and transform both sets, using the required 2D array reshaping syntax.

In [13]:
# Define all continuous features, then filter for those that were selected
continuous_features = [
    'age', 'yrs_experience', 'family_size', 'income', 'mortgage_amt', 
    'credit_card_acct', 'credit_card_spend', 'share_trading_acct', 
    'fixed_deposit_acct', 'online_acct'
]
cols_to_scale = [col for col in continuous_features if col in selected_features]

# Scale selected continuous columns
for col in cols_to_scale:
    scaler = StandardScaler()
    
    # Fit strictly on training data with 2D reshape
    scaler.fit(X_train_selected[col].values.reshape(-1, 1))
    
    # Transform both train and test sets
    X_train_selected[col] = scaler.transform(X_train_selected[col].values.reshape(-1, 1)).flatten()
    X_test_selected[col] = scaler.transform(X_test_selected[col].values.reshape(-1, 1)).flatten()

## 6. Train Baseline Model (Random Forest)
With the data rigorously prepared, we will now instantiate and train our baseline Random Forest classifier.

In [14]:
# Instantiate and fit the baseline Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_selected, y_train)

# Generate predictions on the test set
y_pred = rf_model.predict(X_test_selected)

## 7. Model Evaluation
Because of the class imbalance inherent in loan acceptance data, accuracy is an invalid metric. We will evaluate our model strictly using a flattened Confusion Matrix, Precision, Recall, and the F1-Score.

In [15]:
# Flatten the confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Calculate specific metrics for imbalanced classes
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Output the evaluation metrics
print("Baseline Random Forest Evaluation")
print("*" * 35)
print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")
print("*" * 35)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

Baseline Random Forest Evaluation
***********************************
True Negatives (TN):  967
False Positives (FP): 53
False Negatives (FN): 58
True Positives (TP):  122
***********************************
Precision: 0.6971
Recall:    0.6778
F1-Score:  0.6873
